<a href="https://colab.research.google.com/github/MellyMiranda/jarvis-academico/blob/main/Trabalho_jarvis_academico.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### **Jarvis Acadêmico**

In [1]:
# =========================
# IMPORTS E CONFIGURAÇÕES
# =========================

In [2]:
!apt-get install git
!pip install PyPDF2
!pip install sentence-transformers
!pip install scikit-learn
!pip install openai
!pip install gradio

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git is already the newest version (1:2.34.1-1ubuntu1.17).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.


In [3]:
from PyPDF2 import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import os
import json

In [4]:
import gradio as gr
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo

In [5]:
!git clone https://github.com/MellyMiranda/jarvis-academico.git
pasta_pdfs = "jarvis-academico/data"

fatal: destination path 'jarvis-academico' already exists and is not an empty directory.


In [6]:
client = OpenAI(base_url='https://llm.liaufms.org/v1/gemma-3-12b-it', api_key='Cxt2ftLF7d3mHS2JdiFqB-eSDAQeZvFATPXPs02lV9A')
resp = client.chat.completions.create(
    model='google/gemma-3-12b-it',
    messages=[{'role': 'user', 'content': 'Hi'}],
)
print(resp.choices[0].message.content)


Hi there! 😊 How can I help you today? 



Just let me know what you'd like to talk about, ask, or do.


In [7]:
# =========================
# RAG
# -carrega PDFs
# -tranforma em chunking
# -embeddings
# -resposta coerente
# =========================

In [8]:
documentos = []
for arquivo in os.listdir(pasta_pdfs): ##lendo pdfs

    if arquivo.endswith(".pdf"):

        caminho_completo = os.path.join(pasta_pdfs, arquivo)

        reader = PdfReader(caminho_completo)

        texto = ""

        for pagina in reader.pages:
            texto += pagina.extract_text()

        documentos.append({
            "arquivo": arquivo,
            "texto": texto
        })

        print(f"{arquivo} carregado com sucesso!")

2.IA-introducao.pdf carregado com sucesso!
4.RegressãoLogística.pdf carregado com sucesso!
6.IA e o seu impacto na otimização de processo.pdf carregado com sucesso!
7.ConceitosIA.pdf carregado com sucesso!
10.ML e DL.pdf carregado com sucesso!
9.Embeddings.pdf carregado com sucesso!
5.RedesNeurais.pdf carregado com sucesso!
3.ia_intro.pdf carregado com sucesso!
1.Fundamentos de Machine Learning.pdf carregado com sucesso!
8.Rag.pdf carregado com sucesso!


In [9]:
chunks = []
tamanho_chunk = 500
overlap = 100
passo = tamanho_chunk - overlap

In [10]:
for doc in documentos:

    texto = doc["texto"]

    for i in range(0, len(texto), passo):

        chunk = texto[i:i + tamanho_chunk]

        chunks.append({
            "arquivo": doc["arquivo"],
            "texto": chunk
        })

In [11]:
modelo_embedding = SentenceTransformer('all-MiniLM-L6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [12]:
for chunk in chunks:

    embedding = modelo_embedding.encode(chunk["texto"])

    chunk["embedding"] = embedding

In [13]:
def buscar_chunks(pergunta, top_k=3):

    embedding_pergunta = modelo_embedding.encode(pergunta)

    similaridades = []

    for chunk in chunks:

        similaridade = cosine_similarity(
            [embedding_pergunta],
            [chunk["embedding"]]
        )[0][0]

        similaridades.append({
            "arquivo": chunk["arquivo"],
            "texto": chunk["texto"],
            "similaridade": similaridade
        })

    similaridades = sorted(
        similaridades,
        key=lambda x: x["similaridade"],
        reverse=True
    )

    return similaridades[:top_k]

In [14]:
##Integracao Retrieval com a Gemma 12B

In [15]:
def montar_contexto(resultados):

    contexto = ""

    for resultado in resultados:

        contexto += resultado["texto"]
        contexto += "\n\n"

    return contexto

In [16]:
def responder_pergunta(pergunta):

    resultados = buscar_chunks(pergunta)

    contexto = montar_contexto(resultados)

    prompt = f"""
Você é um assistente acadêmico especializado em Inteligência Artificial e Machine Learning.

Responda a pergunta do usuário utilizando apenas o contexto fornecido.

Explique de maneira clara, organizada e didática.

Se a informação não estiver presente no contexto, diga que não encontrou informações suficientes.

Contexto:
{contexto}

Pergunta:
{pergunta}
"""

    resposta = client.chat.completions.create(
        model='google/gemma-3-12b-it',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ]
    )

    return resposta.choices[0].message.content

In [17]:
def buscar_material_rag(pergunta):
    return responder_pergunta(pergunta)

In [18]:
##TESTE-------------------------------
resposta = responder_pergunta(
    "Qual a diferença entre classificação e regressão?"
)

print(resposta)

De acordo com o contexto fornecido, a diferença entre classificação e regressão é a seguinte:

*   **Classificação:** Tem como resultado uma saída **categórica/discreta**.
*   **Regressão:** Tem como resultado uma saída **numérica**.

Em outras palavras, a classificação atribui dados a categorias, enquanto a regressão prevê valores numéricos.


In [19]:
# ========================= MINIMO 5 OPCOES
# AGENDA ACADÊMICA: (implementadas)
# listar_agenda
# listar_tarefas
# adicionar_tarefa
# adicionar_evento(agenda)
# excluir tarefa
# excluir evento
# concluir_tarefa
# =========================

In [20]:
agenda = []
tarefas = []
logs = []

In [21]:
# =========================
# 1 - TAREFAS
# =========================

In [22]:
def salvar_tarefas():

    with open("jarvis-academico/tarefas.json", "w", encoding="utf-8") as arquivo:

        json.dump(
            tarefas,
            arquivo,
            ensure_ascii=False,
            indent=4
        )

In [23]:
def carregar_tarefas():

    global tarefas

    try:

        with open("jarvis-academico/tarefas.json", "r", encoding="utf-8") as arquivo:

            tarefas = json.load(arquivo)

        print("Tarefas carregadas com sucesso!")

    except:

        tarefas = []

        print("Nenhum arquivo de tarefas encontrado.")

In [24]:
carregar_tarefas()

Tarefas carregadas com sucesso!


In [25]:
def adicionar_tarefa(titulo, descricao):
    if not titulo:
        return "Erro: título não informado."

    tarefa = {
        "titulo": titulo,
        "descricao": descricao,
        "concluida": False
    }

    tarefas.append(tarefa)

    salvar_tarefas()

    return "Tarefa adicionada com sucesso!"

In [26]:
def listar_tarefas():

    if len(tarefas) == 0:
        return "Nenhuma tarefa cadastrada."

    texto = ""

    for i, tarefa in enumerate(tarefas):

        status = "✅ Concluída" if tarefa["concluida"] else "❌ Pendente"

        texto += f"""
ID: {i}
Título: {tarefa['titulo']}
Descrição: {tarefa['descricao']}
Status: {status}

"""

    return texto

In [27]:
def excluir_tarefa(id_tarefa):

    if 0 <= id_tarefa < len(tarefas):

        tarefas.pop(id_tarefa)
        salvar_tarefas()

        return "Tarefa excluída com sucesso!"

    return "ID inválido!"

In [28]:
def concluir_tarefa(id_tarefa):

    if 0 <= id_tarefa < len(tarefas):

        tarefas[id_tarefa]["concluida"] = True

        salvar_tarefas()

        return "Tarefa concluída com sucesso!"

    return "ID inválido!"

In [29]:
# =========================
# 2 - AGENDA
# =========================

In [73]:
def salvar_agenda():

    with open("jarvis-academico/agenda.json", "w", encoding="utf-8") as f:
        json.dump(agenda, f, ensure_ascii=False, indent=4)

In [77]:
def carregar_agenda():

    global agenda

    try:
        with open("jarvis-academico/agenda.json", "r", encoding="utf-8") as f:
            agenda = json.load(f)

        print("Agenda carregada com sucesso!")

    except:
        agenda = []
        print("Nenhum arquivo de agenda encontrado.")

In [99]:
carregar_agenda()

Agenda carregada com sucesso!


In [100]:
def listar_agenda(periodo="todos"):

  hoje = datetime.now(ZoneInfo("America/Campo_Grande"))

  eventos_filtrados = []

  for evento in agenda:

      # FORMATO: 20/05/2026
      data_evento = datetime.strptime(
          evento["data"],
          "%d/%m/%Y"
      )

      # HOJE
      if periodo == "hoje":

          if data_evento.date() == hoje.date():

              eventos_filtrados.append(evento)

      # AMANHÃ
      elif periodo == "amanha":

          amanha = hoje + timedelta(days=1)

          if data_evento.date() == amanha.date():

              eventos_filtrados.append(evento)

      # SEMANA
      elif periodo == "semana":

          diferenca = (data_evento.date() - hoje.date()).days

          if 0 <= diferenca <= 7:

              eventos_filtrados.append(evento)

      # TODOS
      else:

          eventos_filtrados.append(evento)

  if len(eventos_filtrados) == 0:

      return "Nenhum evento encontrado."

  texto = ""

  for evento in eventos_filtrados:

      texto += f"""

  ID: {evento['id']}
  Data: {evento['data']}
  Local: {evento['local']}
  Descrição: {evento['descricao']}

  """

  return texto


In [34]:
def adicionar_evento(data, local, descricao):

    if not data:
        return "Erro: data não informada."

    if not local:
        return "Erro: local não informado."

    evento = {
        "id": len(agenda),
        "data": data,
        "local": local,
        "descricao": descricao
    }

    agenda.append(evento)

    salvar_agenda()

    return "Evento adicionado com sucesso!"

In [35]:
def excluir_evento(id_evento):

    if 0 <= id_evento < len(agenda):

        agenda.pop(id_evento)
        salvar_agenda()

        return "Evento excluído com sucesso!"

    return "ID inválido!"

In [36]:
# =========================
# LOGS
# =========================

In [37]:
def ver_logs():

    if len(logs) == 0:
        return "Nenhum log ainda."

    texto = ""

    for log in logs:

        texto += f"""
        Horário: {log['horario']}
        Tool: {log['tool']}
        Entrada:{log['entrada']}
        Saída:{log['saida']}
        Status: {log['status']}

        ==============================
        """

    return texto

In [38]:
def salvar_logs():

    with open("jarvis-academico/logs.json", "w", encoding="utf-8") as arquivo:

        json.dump(
            logs,
            arquivo,
            ensure_ascii=False,
            indent=4
        )

In [39]:
def carregar_logs():

    global logs

    try:

        with open("jarvis-academico/logs.json", "r", encoding="utf-8") as arquivo:

            logs = json.load(arquivo)

        print("Logs carregados com sucesso!")

    except:

        logs = []

        print("Nenhum arquivo de logs encontrado.")

In [40]:
carregar_logs()

Logs carregados com sucesso!


In [41]:
# =========================
# TOOL CALLING - IA decide qual função usar.
# =========================

In [42]:
tools = {
    #LISTAGENS
    "listar_tarefas": listar_tarefas,
    "listar_agenda": listar_agenda,
    #EXCLUSOES
    "excluir_tarefa": excluir_tarefa,
    "excluir_evento": excluir_evento,
    #ADICOES
    "adicionar_evento": adicionar_evento,
    "adicionar_tarefa": adicionar_tarefa,
    #CONCLUSOES
    "concluir_tarefa": concluir_tarefa,
    #BUSCA
    "buscar_material_rag": buscar_material_rag,
}

In [43]:
def escolher_tool(pergunta):

    prompt = f"""
Você é um sistema de seleção de ferramentas.

Escolha apenas UMA tool da lista abaixo.

Tools disponíveis:
- listar_tarefas
- listar_agenda
- buscar_material_rag
- concluir_tarefa
- excluir_tarefa
- excluir_evento
- adicionar_tarefa
- adicionar_evento


Regras:
- perguntas acadêmicas -> buscar_material_rag
- tarefas -> listar_tarefas
- agenda -> listar_agenda
- concluir tarefa -> concluir_tarefa
- excluir tarefa -> excluir_tarefa
- excluir evento -> excluir_evento
- adicionar tarefa -> adicionar_tarefa
- adicionar evento -> adicionar_evento

FORMATO DE RESPOSTA (OBRIGATÓRIO):

Responda APENAS com o nome da tool.

Use:
- buscar_material_rag para perguntas sobre conteúdos acadêmicos, IA, Machine Learning, regressão, embeddings e materiais de estudo.
- listar_tarefas para tarefas.
- listar_agenda para agenda e eventos.
- concluir_tarefa para concluir tarefas.
- excluir_tarefa para excluir tarefas.
- excluir_evento para excluir eventos.
- adicionar_tarefa para adicionar tarefas.
- adicionar_evento para adicionar eventos.

Responda APENAS com o nome da tool.

Pergunta:
{pergunta}
"""

    resposta = client.chat.completions.create(
        model='google/gemma-3-12b-it',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ]
    )

    return resposta.choices[0].message.content.strip()

In [44]:
def executar_tool(nome_tool, analise):

    # ERRO DE JSON / COMANDO
    if nome_tool == "erro":
        return analise["mensagem"]

    if nome_tool in tools:

        try:

            # RAG
            if nome_tool == "buscar_material_rag":
                resultado = tools[nome_tool](analise.get("pergunta", ""))

            # CONSULTA AGENDA
            elif nome_tool == "listar_agenda":
                resultado = tools[nome_tool](
                analise.get("periodo", "todos")
                )

            # TAREFA / AGENDA COM ID
            elif nome_tool in ["concluir_tarefa", "excluir_tarefa", "excluir_evento"]:
                id_item = analise.get("id") or analise.get("ID") or analise.get("numero") or 0
                resultado = tools[nome_tool](id_item)

            # ADICIONAR TAREFA
            elif nome_tool == "adicionar_tarefa":
                resultado = tools[nome_tool](
                    analise.get("titulo"),
                    analise.get("descricao")
                )

            # ADICIONAR EVENTO
            elif nome_tool == "adicionar_evento":
                resultado = tools[nome_tool](
                    analise.get("data"),
                    analise.get("local"),
                    analise.get("descricao")
                )

            # LISTAGENS
            else:
                resultado = tools[nome_tool]()

            # LOG
            log = {
              "horario": str(datetime.now()),
              "tool": nome_tool,
              "entrada": analise,
              "saida": resultado,
              "status": "sucesso"
            }

            logs.append(log)
            salvar_logs()

            return resultado

        except Exception as e:
          log = {
              "horario": str(datetime.now()),
              "tool": nome_tool,
              "entrada": analise,
              "saida": str(e),
              "status": "erro"
          }

          logs.append(log)
          salvar_logs()

          return f"Erro ao executar tool: {e}"

    return "Tool não encontrada"

In [45]:
def analisar_pergunta(pergunta):

    prompt = f"""
Você é um sistema de roteamento de ferramentas.

Sua tarefa é escolher a ferramenta correta e extrair parâmetros quando necessário.

TOOLS DISPONÍVEIS:
- listar_tarefas
- listar_agenda
- buscar_material_rag
- concluir_tarefa
- excluir_tarefa
- excluir_evento
- adicionar_tarefa
- adicionar_evento

REGRAS:
- perguntas acadêmicas -> buscar_material_rag
- tarefas -> listar_tarefas
- agenda -> listar_agenda
- concluir tarefa -> concluir_tarefa
- excluir tarefa -> excluir_tarefa
- excluir evento -> excluir_evento
- adicionar tarefa -> adicionar_tarefa
- adicionar evento -> adicionar_evento

FORMATO DE RESPOSTA:
Responda SOMENTE em JSON válido.

EXEMPLOS:

Pergunta: Concluir tarefa 2
{{
  "tool": "concluir_tarefa",
  "id": 2
}}

Pergunta: Excluir evento 1
{{
  "tool": "excluir_evento",
  "id": 1
}}

Pergunta: Adicionar tarefa estudar IA, domingo
{{
  "tool": "adicionar_tarefa",
  "titulo": "estudar IA",
  "descricao":"domingo"
}}

Pergunta: Adicionar evento 20/05/2026, LAB 1, maratona prog
{{
  "tool": "adicionar_evento",
  "data": "20/05/2026",
  "local": "LAB 1",
  "descricao": "maratona prog"
}}

Pergunta: O que é regressão logística?
{{
  "tool": "buscar_material_rag",
  "pergunta": "O que é regressão logística?"
}}

Pergunta: O que tenho hoje?
{{
  "tool": "listar_agenda",
  "periodo": "hoje"
}}
Pergunta: Tenho prova amanhã?
{{
  "tool": "listar_agenda",
  "periodo": "amanha"
}}
Pergunta: Quais meus eventos da semana?
{{
  "tool": "listar_agenda",
  "periodo": "semana"
}}
Pergunta: Liste minha agenda
{{
  "tool": "listar_agenda",
  "periodo": "todos"
}}


Agora responda:
Pergunta: {pergunta}
"""

    resposta = client.chat.completions.create(
        model='google/gemma-3-12b-it',
        messages=[
            {
                'role': 'user',
                'content': prompt
            }
        ],
        temperature=0
    )

    texto = resposta.choices[0].message.content.strip()

    inicio = texto.find("{")
    fim = texto.rfind("}") + 1

    texto_json = texto[inicio:fim]

    try:
      return json.loads(texto_json)

    except Exception:
      return {"tool": "erro", "mensagem": "Não consegui entender o comando."
      }

In [46]:
def executar_comando(pergunta):

    analise = analisar_pergunta(pergunta)

    nome_tool = analise["tool"]

    return executar_tool(nome_tool, analise)

In [47]:
def jarvis(pergunta):

    try:

        resposta = executar_comando(pergunta)

        return resposta

    except Exception as erro:

        return f"Erro: {erro}"

In [48]:
# =========================
# TESTE
# =========================

In [49]:

def teste_geral():

    print("\n========================")
    print("🧠 TESTE RAG")
    print("========================")
    print(jarvis("O que é regressão logística?"))

    print("\n========================")
    print("📋 TESTE ADICIONAR TAREFA")
    print("========================")
    print(jarvis("Adicionar tarefa estudar IA SABADO"))
    print(jarvis("Adicionar tarefa estudar IA DOMINGO"))

    print("\n========================")
    print("📋 TESTE LISTAR TAREFAS")
    print("========================")
    print(jarvis("Listar minhas tarefas"))

    print("\n========================")
    print("✅ TESTE CONCLUIR TAREFA")
    print("========================")
    print(jarvis("Concluir tarefa 1"))

    print("\n========================")
    print("🗑️ TESTE EXCLUIR TAREFA")
    print("========================")
    print(jarvis("Excluir tarefa 0"))
    print("\n========================")
    print("✅ TESTE MOSTRAR TAREFAS APOS EXCLUSAO")
    print("========================")
    print(jarvis("Listar minhas tarefas"))

    print("\n========================")
    print("📅 TESTE ADICIONAR EVENTO")
    print("========================")
    print(jarvis("Adicionar evento 2026-05-20, prova MNN, nivel MEdio"))

    print("\n========================")
    print("📅 TESTE LISTAR AGENDA")
    print("========================")
    print(jarvis("Listar agenda"))

    print("\n========================")
    print("🗑️ TESTE EXCLUIR EVENTO")
    print("========================")
    print(jarvis("Excluir evento 0"))

    print("\n========================")
    print("✅ TESTE MOSTRAR TAREFAS APOS EXCLUSAO")
    print("========================")
    print(jarvis("Listar minhas agenda"))
    print("\n========================")
    print("📊 TESTE FINAL COMPLETO")
    print("========================")


teste_geral()


🧠 TESTE RAG
De acordo com o contexto fornecido:

A regressão logística é um modelo que utiliza um conjunto de variáveis independentes (fatores explicativos) que compõem o preditor linear do modelo, e uma função de ligação entre a média da variável resposta e o referido preditor linear. O contexto menciona o modelo binomial logit e o modelo multinomial logit, que podem ser vistos como extensões do modelo binário logit.

Além disso, a aprendizagem supervisionada, que pode ser utilizada para resolver problemas de classificação (com saída categórica/discreta) e regressão (com saída numérica), inclui a regressão linear como um dos algoritmos.

📋 TESTE ADICIONAR TAREFA
Tarefa adicionada com sucesso!
Tarefa adicionada com sucesso!

📋 TESTE LISTAR TAREFAS

ID: 0
Título: estudar IA
Descrição: DOMINGO
Status: ✅ Concluída


ID: 1
Título: estudar IA
Descrição: SABADO
Status: ❌ Pendente


ID: 2
Título: estudar IA
Descrição: DOMINGO
Status: ❌ Pendente



✅ TESTE CONCLUIR TAREFA
Tarefa concluída com

In [50]:
# =========================
# INTERFACE (GRADIO)
# =========================

In [51]:
def chat_interface(mensagem, historico):

    resposta = jarvis(mensagem)

    historico.append({
        "role": "user",
        "content": mensagem
    })

    historico.append({
        "role": "assistant",
        "content": resposta
    })

    return historico, historico, ""

In [52]:
def limpar_chat():
    return []

In [53]:
def mensagem_inicial():

    return [
        {
            "role": "assistant",
            "content": """🤖 JARVIS ACADÊMICO

📌 MENU E COMO USAR (IMPORTANTE):

🟢 TAREFAS
Adicionar Tarefa:
→ Modelo: titulo, descricao

Listar Tarefas:
→ Modelo: Listar minhas tarefas

Concluir uma Tarefa:
→ Modelo: Concluir tarefa 0 (USE ID)

Excluir Tarefa:
→ Modelo: Excluir tarefa 0 (USE ID)

────────────────────

🔵 AGENDA
Adicionar Evento:
→ Modelo: data, local, descricao

Listar Agenda:
→ Modelo: Listar agenda

Eventos Hoje:
→ Modelo: O que tenho hoje?

Eventos Amanhã:
→ Modelo: Tenho algo amanhã?

Eventos da Semana:
→ Modelo: Quais meus eventos da semana?

Excluir Evento:
→ Modelo: Excluir evento 0 (USE ID)

────────────────────

📚 RAG (MATERIAIS)
→ O que é regressão logística?
→ Explique embeddings
→ Resuma redes neurais

────────────────────
"""
        }
    ]

In [80]:
css = """
body {
    font-size: 14px;
}

.gradio-container {
    max-width: 1400px !important;
}

.message {
    font-size: 14px !important;
}

textarea {
    font-size: 14px !important;
}
"""

with gr.Blocks(
    theme=gr.themes.Soft(),
    css=css
) as demo:

    gr.Markdown("# 🤖 JARVIS ACADÊMICO")

    # =========================
    # CHAT
    # =========================

    with gr.Tab("💬 Chat"):

        chatbot = gr.Chatbot(
            type="messages",
            height=500,
            bubble_full_width=False,
            value=mensagem_inicial()
        )

        msg = gr.Textbox(
            label="Digite sua pergunta",
            placeholder="Ex: Adicionar tarefa, estudar IA, revisar embeddings",
            lines=1
        )

        state = gr.State([])


        with gr.Row():
          send = gr.Button("Enviar",variant="primary")

        menu_btn = gr.Button("MENU/DÚVIDAS")

        clear = gr.Button("Limpar chat")

        send.click(
            chat_interface,
            [msg, state],
            [chatbot, state, msg]
        )

        msg.submit(chat_interface,[msg, state],[chatbot, state, msg])

        clear.click(limpar_chat,outputs=[chatbot])
        menu_btn.click(mensagem_inicial,outputs=[chatbot])

    # =========================
    # TAREFAS
    # =========================

    with gr.Tab("📋 Tarefas"):

        tarefas_btn = gr.Button("Atualizar tarefas")

        tarefas_out = gr.Textbox(lines=15)

        tarefas_btn.click(
            lambda: listar_tarefas(),
            outputs=[tarefas_out]
        )

    # =========================
    # AGENDA
    # =========================

    with gr.Tab("📅 Agenda"):

        agenda_btn = gr.Button("Atualizar agenda")

        agenda_out = gr.Textbox(lines=15)

        agenda_btn.click(
            lambda: listar_agenda(),
            outputs=[agenda_out]
        )

    # =========================
    # LOGS
    # =========================

    with gr.Tab("📊 Logs"):

        logs_btn = gr.Button("Ver logs")

        logs_out = gr.Textbox(lines=20)

        logs_btn.click(
            lambda: ver_logs(),
            outputs=[logs_out]
        )


/tmp/ipykernel_17671/1252152138.py:19: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_17671/1252152138.py:19: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(
/tmp/ipykernel_17671/1252152138.py:32: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(
/tmp/ipykernel_17671/1252152138.py:32: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


In [55]:
# =========================
# EXECUÇÃO
# =========================
demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d6d7249d2b2440ab92.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
